In [20]:
from passwords import *
import pandas as pd
try:
    import boto3
except:
    !pip install boto3
    import boto3

import os
import pyodbc

#### Functions

In [25]:
# write function to push local files into S3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # initialize boto3 client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    
    cls_client.upload_file(
        str_local_path,
        str_bucket_name,
        str_bucket_key
    )

#### Constants

In [13]:
# project
str_project = os.getcwd().split('\\')[5].replace('_','-')
print(str_project)

# task
str_task = os.getcwd().split('\\')[6]
print(str_task)

# sub task
str_subtask = os.getcwd().split('\\')[7]
print(str_subtask)

20240521-bridger-internship
03_python
01_jupyter_practice


#### Make output directory

In [15]:
try:
    os.mkdir('output')
except:
    pass

#### Read in a sql query and get dataframe

In [17]:
# get our query
str_query = open('./sql/GenScoresTable2024.sql', 'r').read()
print(str_query)

SELECT tmp.bigAccountId, tmp.bigDealerId, acc.dtmStampCreation, tmp.dtmFunded, tmp.intTerm, tmp.intOpenBKType, tmp.strTier, tmp.AmtFinanced, tmp.OriginalInterestRate, tmp.MaxFico,

MAX(CASE WHEN dim.strScoreCardVersion LIKE 'genxi%' THEN dim.fltDebtorScore END) AS 'Gen 11',
MAX(CASE WHEN dim.strScoreCardVersion LIKE 'genxii%' THEN dim.fltDebtorScore END) AS 'Gen 12'

FROM electra.riskdb.analytics.tbltempstaticpool tmp LEFT JOIN electra.pfsdb.dbo.tblaccount acc
ON tmp.bigAccountId = acc.bigAccountId LEFT JOIN 
	(SELECT dim1.intAccountKey, dim1.fltdebtorscore, dim1.strscorecardversion, ROW_NUMBER() OVER 
		(PARTITION BY dim1.intAccountKey, dim1.strScoreCardVersion
		ORDER BY dim1.dtmstampcreation DESC) as rn FROM edw.pfsedw.dbo.DimScoreCard dim1) dim 
ON tmp.bigAccountId = dim.intAccountkey AND dim.rn = 1 
WHERE acc.dtmStampCreation >= '2024-01-01'
GROUP BY tmp.bigAccountId, tmp.bigDealerId, acc.dtmStampCreation, tmp.dtmFunded, tmp.intTerm, tmp.intOpenBKType, tmp.strTier, tmp.AmtFinanced

In [21]:
# read it in and display as dataframe
# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)

df = pd.read_sql_query(
    str_query,
    con=conn,
)

# close connection
conn.close()

<ipython-input-21-8c76d232cac2>:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


#### Save dataframe locally as GZIP parquet

In [29]:
str_filename = 'df_model_scores.gzip'
str_local_path = f'./output/{str_filename}'
df.to_parquet(str_local_path,
             index=False,
             compression='gzip')

#### Upload to S3

In [30]:
%%time

upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    str_local_path=str_local_path,
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}',
    str_bucket_name=str_project
)

Wall time: 1.43 s
